# 11 — Induction-ordering (H1) vs Set-difference (H2)

How does the LM pick the next sub-task at the pivot? Two hypotheses:
- **H1 (induction ordering):** each task stores its predecessor ("I come after C"); the pivot query is "what comes after the frontier".
- **H2 (set-difference / DIFF):** the query carries progress (done-set); the next task is the set-complement.

Contrastive pair (`src.diff.h1h2`), five roles A/C/D/B/E:
```
SRC.tasks=[A,C,D,B]  prefill={A,C,B}  targ=D   # D = successor-of-C (H1) AND set-complement (H2); gap at D rules out counting
DST.tasks=[A,C,B,E]  prefill={A}      targ=C   # native
```
Patching SRC pivot query into DST later splits them: **H1→B, H2→E, COPY→D, NO-OP→C**. Prompts use the plain `computes:` instruction (no "in sequential order") and positional letters. This notebook builds one pair and inspects the **source** and the **clean destination** runs.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append("../..")
import torch
import src.diff.functional as FUNC
from src.diff.functional import *          # rich_print, greedy, goalscope_readout, ...
import src.diff.data as DATA
import src.diff.h1h2 as H
from src.tokens import prepare_input
from rich.markup import escape
FUNC.setup("Qwen/Qwen2.5-Coder-32B-Instruct")   # loads model + goal-head mask
mt = FUNC.mt
GOAL_HEADS = READOUT_HEADS = FUNC.GOAL_HEADS
torch.set_grad_enabled(False)
pivot_of = lambda prompt: prepare_input(prompts=prompt, tokenizer=mt)['input_ids'].shape[1] - 1

2026-09-11 23:50:24 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


In [2]:
# ── pick a contrastive pair (sample deterministically, or hand-pick roles below) ──
DT = "list[str]"
spec = H.sample_pairs(DT, 1, seed=0)[0]
# spec = H.H1H2Spec(DT, anchor=..., frontier=..., src_targ=..., dst_h1=..., dst_targ=...)   # hand-pick
SRC_TARG_IN_DST = True   # append D to DST -> COPY (already-decided-D) gets a gaze target; splits complement into {E,D}
P    = H.build_prompts(mt, spec, stub="var_eq",  src_targ_in_dst=SRC_TARG_IN_DST)   # pivot pinned at `{var} =`
Pind = H.build_prompts(mt, spec, stub="indent",  src_targ_in_dst=SRC_TARG_IN_DST)   # decision-point pivot

by = {p.id: p for p in DATA.QTASKS[DT].predicates}
lines = [f"[bold]datatype[/bold] {DT}  ·  roles (A=anchor C=frontier D=src_targ B=dst_h1 E=dst_targ):"]
lines += [f"  {r:9s} = [cyan]{getattr(spec, r):24s}[/cyan] {escape(by[getattr(spec, r)].description)}" for r in H.ROLES]
lines.append("\n[bold]expected outcome -> which condition:[/bold]")
lines += [f"  {k:22s}  {pid:24s} {escape(d)}" for k, (role, pid, d) in P["outcomes"].items()]
lines.append(f"\n[bold]SRC_TARG_IN_DST[/bold] = {SRC_TARG_IN_DST}  ->  DST tasks = {[p.id for p in P['dst_inst'].task.predicates]}")
rich_print("\n".join(lines))

datatype list  ·  roles (A=anchor C=frontier D=src_targ B=dst_h1 E=dst_targ):
  anchor    = count_starts_vowel       the number of words that start with a vowel
  frontier  = total_uppercase_letters  the total number of uppercase letters across all words
  src_targ  = count_with_digit         the number of words that contain a digit
  dst_h1    = any_empty                whether any word is empty
  dst_targ  = has_palindrome           whether any word is a palindrome

expected outcome -> which condition:
  H1 (induction)  -> B    any_empty                whether any word is empty
  H2 (set-diff)   -> E    has_palindrome           whether any word is a palindrome
  COPY            -> D    count_with_digit         the number of words that contain a digit
  NO-OP (native)  -> C    total_uppercase_letters  the total number of uppercase letters across all words

SRC_TARG_IN_DST = True  ->  DST tasks = ['count_starts_vowel', 'total_uppercase_letters', 'any_empty', 
'has_palindrome', 'count_with_digit']

## Source run  (SRC = [A, C, D, B], prefill {A, B, C}, target D)

In [3]:
# ── SOURCE run: greedy + goalscope readout, at var_eq (pinned to D) and indent (decision point) ──
stok = prepare_input(prompts=P["src_prompt"], tokenizer=mt)
sind = prepare_input(prompts=Pind["src_prompt"], tokenizer=mt)
rich_print("\n".join([
    "[bold red1]SRC prompt (var_eq — pinned to the target D):[/bold red1]",
    escape(P["src_prompt"]),
    f"[dim]SRC greedy (should implement D = {escape(P['src_targ_desc'])}):[/dim]",
    escape(greedy(stok, max_new=64)),
    "\n[bold red1]SRC (indent — model CHOOSES the next variable; the decision point):[/bold red1]",
    f"[dim]SRC greedy from indent:[/dim]",
    escape(greedy(sind, max_new=64)),
]))
goalscope_readout("   ↳ goalscope · SRC @ pivot", P["src_prompt"], pivot_of(P["src_prompt"]), heads=READOUT_HEADS)

SRC prompt (var_eq — pinned to the target D):
<|im_start|>system
You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just the 
code, no need to comment or explain.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `func` which, given a list of strings `words`, computes:
(1) the number of words that start with a vowel
(2) the total number of uppercase letters across all words
(3) the number of words that contain a digit
(4) whether any word is empty
<|im_end|>
<|im_start|>assistant
<code>
def func(words: list[str]):
    a = sum(1 for w in words if w and w[0].lower() in "aeiou")
    b = any(len(w) == 0 for w in words)
    c = sum(1 for w in words for c in w if c.isupper())
    d =
SRC greedy (should implement D = the number of words that contain a digit):
 sum(1 for w in words if any(c.isdigit() for c in w))
    return a, c, d, b
</code>

SRC (indent — model CHOOSES the next variable; the decision point):
SRC greedy from indent:
 d = sum(1 for w in words if any(c.isdigit() for c in w))
    return a, c, d, b
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


   ↳ goalscope · SRC @ pivot  pos=185 ' =' => I'm checking if any character in the string is a digit.

[{'pos': 185,
  'token': ' =',
  'patched_gen': "I'm checking if any character in the string is a digit.",
  'scoped': True}]

## Clean destination run  (DST = [A, C, B, E], prefill {A}, native target C)

In [4]:
# ── CLEAN DEST run: greedy + goalscope readout, at var_eq (native C) and indent (decision point) ──
dtok = prepare_input(prompts=P["dst_prompt"], tokenizer=mt)
dind = prepare_input(prompts=Pind["dst_prompt"], tokenizer=mt)
rich_print("\n".join([
    "[bold sea_green3]DST prompt (var_eq — pinned to native C):[/bold sea_green3]",
    escape(P["dst_prompt"]),
    f"[dim]DST greedy (native, should implement C = {escape(P['dst_native_desc'])}):[/dim]",
    escape(greedy(dtok, max_new=64)),
    "\n[bold sea_green3]DST (indent — model CHOOSES the next variable):[/bold sea_green3]",
    f"[dim]DST greedy from indent:[/dim]",
    escape(greedy(dind, max_new=64)),
]))
goalscope_readout("   ↳ goalscope · DST @ pivot", P["dst_prompt"], pivot_of(P["dst_prompt"]), heads=READOUT_HEADS)

DST prompt (var_eq — pinned to native C):
<|im_start|>system
You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just the 
code, no need to comment or explain.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `func` which, given a list of strings `words`, computes:
(1) the number of words that start with a vowel
(2) the total number of uppercase letters across all words
(3) whether any word is empty
(4) whether any word is a palindrome
(5) the number of words that contain a digit
<|im_end|>
<|im_start|>assistant
<code>
def func(words: list[str]):
    a = sum(1 for w in words if w and w[0].lower() in "aeiou")
    b =
DST greedy (native, should implement C = the total number of uppercase letters across all words):
 sum(c.isupper() for w in words for c in w)
    c = any(not w for w in words)
    d = any(w == w[::-1] for w in words)
    e = sum(any(c.isdigit() for c in w) for w in words)
    return a, b,

DST (indent — model CHOOSES the next variable):
DST greedy from indent:
 b = sum(c.isupper() for w in words for c in w)
    c = any(not w for w in words)
    d = any(w == w[::-1] for w in words)
    e = sum(any(c.isdigit() for c in w) for w in words)
    return a,

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · DST @ pivot  pos=161 ' =' => I'm counting uppercase letters in the string.

[{'pos': 161,
  'token': ' =',
  'patched_gen': "I'm counting uppercase letters in the string.",
  'scoped': True}]

## Depth-collapse tools, LAYER-WISE — gaze (Tool 1) + readout (Tool 2)

For a sweep of collapse layers `L`, force all goal heads to read `h^L` and read out, at the pivot:
- **Tool 1 (gaze):** value-weighted pivot->task attention as a **fraction over the task conditions** (each row sums to 1 across the numbered predicates). [green]green[/] = argmax, [yellow]yellow[/] = the target condition — so you can see *by which layer the gaze locks onto the target*.
- **Tool 2 (readout):** the collective OV write, verbalized (last column).

On clean SRC the gaze should converge on **D**; on clean DST, on **C** (D present but low). Later, patch `h_src` into DST and re-run this: the layer where gaze moves to B / E / D tells you H1 / H2 / COPY.

In [5]:
# ── LAYER-WISE depth-collapse: gaze (Tool 1) + readout (Tool 2). Readouts are BATCHED across layers ──
# (one head_additive_generate over all layers' OV-write vectors, like OVReader.apply_probe — NOT one
#  apply_goalscope per layer). Per layer we only do cheap single forwards to extract the write + gaze.
import numpy as np, torch
from rich.table import Table
from scripts.evaluation.eval_collapse_layerwise import resid_all, read_from_layer_iface, collapsed_gaze_alignment
from src.diff.functional import head_write                                   # collective OV write (sum_h o_in[h] @ W_o[:,h])
from src.goals.goalscope import head_additive_generate, _build_goalscope_prompt
from src.tokens import find_token_range

DEMO_LAYERS = [16, 20, 24, 28, 32, 36, 40, 44, 48, 52]   # collapse layers to sweep
STEER_BAND  = (35, 46)                                     # GoalScope steer band for the readout injection
_PROBE = prepare_input(prompts=_build_goalscope_prompt(mt), tokenizer=mt)

def _pred_spans(prompt, inst):
    return [find_token_range(prompt, pr.description, mt.tokenizer) for pr in inst.task.predicates]

def demo_layerwise(tag, prompt, inst, target_id, layers=DEMO_LAYERS, heads=None, max_new=24, alpha=0.1):
    heads = list(READOUT_HEADS if heads is None else heads)
    tok = prepare_input(prompts=prompt, tokenizer=mt); piv = tok["input_ids"].shape[1] - 1
    spans = _pred_spans(prompt, inst); ids = [pr.id for pr in inst.task.predicates]; tgt = ids.index(target_id); T = len(ids)
    # per layer: Tool-1 gaze fracs + the collective OV write vector (real attention under the collapse)
    gaze, steer = [], []
    for L in layers:
        hL = resid_all(mt, tok, L)
        _, _, per = collapsed_gaze_alignment(mt, tok, piv, heads, hL, spans, tgt)
        gaze.append(per)
        steer.append(head_write(prompt, heads, [piv], iface=read_from_layer_iface(mt, hL, heads))[0])  # (n_embd,)
    # Tool 2: ONE batched generation over ALL layers (rank-keyed like OVReader.apply_probe)
    outs = head_additive_generate(mt, _PROBE, torch.stack(steer), layers=list(range(*STEER_BAND)),
                                  max_new_tokens=max_new, strategy="additive_norm", alpha=alpha)
    tbl = Table(title=f"{tag} · collapsed gaze (frac over conditions) + readout · target=({tgt+1}) {target_id}",
                highlight=False)
    tbl.add_column("L", justify="right")
    for i in range(T):
        tbl.add_column(f"({i+1})" + ("*" if i == tgt else ""), justify="right")
    tbl.add_column("Tool 2 readout")
    for L, per, ro in zip(layers, gaze, outs):
        amax = int(np.argmax(per)); tot = sum(per) or 1e-9; cells = []
        for i, m in enumerate(per):
            s = f"{m/tot:.2f}"
            if i == amax:  s = f"[bold green]{s}[/bold green]"
            elif i == tgt: s = f"[yellow]{s}[/yellow]"
            cells.append(s)
        tbl.add_row(str(L), *cells, escape((ro or "").replace(chr(10), " ")[:64]))
    rich_print(tbl)
    rich_print("\n".join(f"   ({i+1}) {escape(pr.description)}"
                         for i, pr in enumerate(inst.task.predicates)) + "\n")

demo_layerwise("SRC (target D)",       P["src_prompt"], P["src_inst"], spec.src_targ)
demo_layerwise("clean DST (native C)", P["dst_prompt"], P["dst_inst"], spec.frontier)

   SRC (target D) · collapsed gaze (frac over conditions) + readout · target=(3) count_with_digit    
┏━━━━┳━━━━━━┳━━━━━━┳━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  L ┃  (1) ┃  (2) ┃ (3)* ┃  (4) ┃ Tool 2 readout                                                   ┃
┡━━━━╇━━━━━━╇━━━━━━╇━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 16 │ 0.39 │ 0.28 │ 0.17 │ 0.16 │ Sure, please provide the line of code you'd like me to analyze.  │
│ 20 │ 0.36 │ 0.25 │ 0.19 │ 0.19 │ Sure, please provide the line of code you'd like me to analyze.  │
│ 24 │ 0.27 │ 0.27 │ 0.24 │ 0.22 │ Sure, please provide the line of code you'd like me to analyze.  │
│ 28 │ 0.19 │ 0.34 │ 0.24 │ 0.24 │ I'm checking if the character is a letter.                       │
│ 32 │ 0.16 │ 0.32 │ 0.28 │ 0.24 │ I'm checking if any character in the string is an uppercase lett │
│ 36 │ 0.09 │ 0.30 │ 0.26 │ 0.35 │ I'm checking if any character in the string is a whitespace.     │
│ 40 │ 0.07 │ 0.24 │ 0.25 │ 0.44 │ I'm checking if any character in the string is a digit.          │
│ 44 │ 0.04 │ 0.10 │ 0.32 │ 0.54 │ I'm checking if any character in the string is a digit.          │
│ 48 │ 0.05 │ 0.11 │ 0.71 │ 0.13 │ I'm checking if any character in the string is a digit.          │
│ 52 │ 0.14 │ 0.18 │ 0.44 │ 0.25 │ I'm checking if any character in the string is a digit.          │
└────┴──────┴──────┴──────┴──────┴──────────────────────────────────────────────────────────────────┘

(1) the number of words that start with a vowel
   (2) the total number of uppercase letters across all words
   (3) the number of words that contain a digit
   (4) whether any word is empty

clean DST (native C) · collapsed gaze (frac over conditions) + readout · target=(2) total_uppercase_letters 
┏━━━━┳━━━━━━┳━━━━━━┳━━━━━━┳━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  L ┃  (1) ┃ (2)* ┃  (3) ┃  (4) ┃  (5) ┃ Tool 2 readout                                                   ┃
┡━━━━╇━━━━━━╇━━━━━━╇━━━━━━╇━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 16 │ 0.35 │ 0.24 │ 0.12 │ 0.14 │ 0.15 │ Sure, please provide the line of code you'd like me to analyze.  │
│ 20 │ 0.34 │ 0.22 │ 0.13 │ 0.14 │ 0.16 │ Sure, please provide the line of code you'd like me to analyze.  │
│ 24 │ 0.29 │ 0.26 │ 0.13 │ 0.15 │ 0.18 │ Sure, but I need to see the line of code first to provide an acc │
│ 28 │ 0.23 │ 0.31 │ 0.14 │ 0.16 │ 0.16 │ I'm checking if the character is a letter and counting it if it  │
│ 32 │ 0.22 │ 0.30 │ 0.12 │ 0.17 │ 0.18 │ I'm checking if the character is a vowel.                        │
│ 36 │ 0.16 │ 0.33 │ 0.11 │ 0.17 │ 0.22 │ I'm checking if any character in the string is a vowel.          │
│ 40 │ 0.14 │ 0.51 │ 0.08 │ 0.12 │ 0.14 │ I'm counting uppercase letters in the string.                    │
│ 44 │ 0.05 │ 0.60 │ 0.06 │ 0.07 │ 0.22 │ I'm counting uppercase letters in the string.                    │
│ 48 │ 0.06 │ 0.72 │ 0.05 │ 0.05 │ 0.12 │ Sure, please provide the line of code you'd like me to analyze.  │
│ 52 │ 0.16 │ 0.38 │ 0.11 │ 0.11 │ 0.24 │ Sure, but I need to see the specific line of code first. Please  │
└────┴──────┴──────┴──────┴──────┴──────┴──────────────────────────────────────────────────────────────────┘

(1) the number of words that start with a vowel
   (2) the total number of uppercase letters across all words
   (3) whether any word is empty
   (4) whether any word is a palindrome
   (5) the number of words that contain a digit

## How many distinct contrastive pairs?

In [ ]:
lines, tot = [], 0
for dt in DATA.DATATYPES:
    c = H.count_pairs(dt); c5 = H.count_pairs(dt, all_nontrivial=True); tot += c
    lines.append(f"{dt:9s}: [bold]{c:>10,d}[/bold]  (all-5-nontrivial: {c5:>10,d})")
lines.append(f"[bold]TOTAL[/bold] (A any, B/C/D/E non-trivial): {tot:,d}")
rich_print("\n".join(lines))

int      :    205,920  (all-5-nontrivial:    154,440)
str      :    524,160  (all-5-nontrivial:    524,160)
list:    142,560  (all-5-nontrivial:     95,040)
list:    524,160  (all-5-nontrivial:    524,160)
TOTAL (A any, B/C/D/E non-trivial): 1,396,800

: 